영화를 보고 남긴 리부를 딥러닝 모델로 학습해서 각 리뷰가 긍정적인지 부정적인지를 예측한다.  
긍정이면 1이라는 클래스를, 부정적이 0이라는 클래스로 지정한다.

In [1]:
from numpy import array

# 텍스트 리뷰 자료를 지정한다.
docs = ['너무 재밌네요', '최고예요', '참 잘 만든 영화예요', '추천하고 싶은 영화입니다.',
        '한번 더 보고싶네요', '글쎄요', '별로예요', '생각보다 지루하네요', '연기가 어색해요',
        '재미없어요']

# 긍정 리뷰는 1, 부정 리뷰는 0으로 클래스를 지정한다.
classes = array([1, 1, 1, 1, 1, 0, 0, 0, 0, 0])

케라스에서 제공하는 Tokenizer() 함수의 fit_on_texts를 이용해 각 단어를 하나의 토큰으로 변환한다.

In [2]:
from tensorflow.keras.preprocessing.text import Tokenizer

# 토큰화
token = Tokenizer()
token.fit_on_texts(docs)
print(token.word_index) # 토큰화된 결과를 출력해 확인한다.

{'너무': 1, '재밌네요': 2, '최고예요': 3, '참': 4, '잘': 5, '만든': 6, '영화예요': 7, '추천하고': 8, '싶은': 9, '영화입니다': 10, '한번': 11, '더': 12, '보고싶네요': 13, '글쎄요': 14, '별로예요': 15, '생각보다': 16, '지루하네요': 17, '연기가': 18, '어색해요': 19, '재미없어요': 20}


토큰에 지정된 인덱스로 새로운 배열을 생성한다.

In [3]:
x = token.texts_to_sequences(docs)
print("\n리뷰 텍스트, 토큰화 결과:\n", x)


리뷰 텍스트, 토큰화 결과:
 [[1, 2], [3], [4, 5, 6, 7], [8, 9, 10], [11, 12, 13], [14], [15], [16, 17], [18, 19], [20]]


딥러닝 모델에 입력하려면 학습 데이터의 길이가 동일해야 한다. 따라서 토큰의 수를 똑같이 맞추어주어야 한다. 이처럼 길이를 똑같이 맞추어 주는 작업을 패딩(padding) 과정이라고 한다.  
  

패딩 작업을 위해 케라스는 pad_sequences() 함수를 제공한다. pad_suquences() 함수를 사용하면 원하는 길이보다 짧은 부분은 숫자 0을 넣어서 채워 주고, 긴 데이터는 잘라서 같은 길이로 맞춘다.

In [4]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

padded_x = pad_sequences(x, 4) # 서로 다른 길이의 데이터를 4로 맞추기
print("\n패딩 결과:\n", padded_x)


패딩 결과:
 [[ 0  0  1  2]
 [ 0  0  0  3]
 [ 4  5  6  7]
 [ 0  8  9 10]
 [ 0 11 12 13]
 [ 0  0  0 14]
 [ 0  0  0 15]
 [ 0  0 16 17]
 [ 0  0 18 19]
 [ 0  0  0 20]]


임베딩 함수에 필요한 세 가지 파라미터는 '입력, 출력, 단어 수'이다. 총 몇 개의 단어 집합에서(입력), 몇 개의 임베딩 결과를 사용할 것인지(출력), 그리고 매번 입력될 단어 수는 몇 개로 할지(단어 수)를 정해야 하는 것이다.  
  
  
먼저 총 몇 개의 인덱스가 '입력'되어야 하는지 정한다. word_size라는 변수를 만든 후 길이를 세는 len() 함수를 이용해 word_index 값을 앞서 만든 변수에 대입한다. 맨 앞에 0이 먼저 나와야 하므로 총 단어 수에 1을 더하는 것을 잊지 말아야한다.

In [5]:
word_size = len(token.word_index) + 1
word_size

21

이제 몇개의 입베딩 결과를 사용할 것인지, 즉 '출력'을 정할 차례읻. 이는 데이터에 따라 임의로 정한 적절한 값는다.  
그 후, 매번 입력될 '단어 수'를 정한다. 패딩 과정을 거쳐 네 개의 길이로 맞추어 주었으므로 네 개의 단어가 들어가게 설정한다.

In [6]:
from tensorflow.keras.layers import Embedding

Embedding(word_size, 8, input_length=4)

In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten

# 단어 임베딩을 포함해 딥러닝 모델을 만들고 결과를 출력한다.
model = Sequential()
model.add(Embedding(word_size, 8, input_length=4))
model.add(Flatten())
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy',
              metrics=['accuracy'])
model.fit(padded_x, classes, epochs=20)
print("\n Accuracy: %.4f" % (model.evaluate(padded_x, classes)[1]))

Epoch 1/20
1/1 [==============================] - 1s 981ms/step - loss: 0.6965 - accuracy: 0.3000
Epoch 2/20
1/1 [==============================] - 0s 12ms/step - loss: 0.6948 - accuracy: 0.4000
Epoch 3/20
1/1 [==============================] - 0s 11ms/step - loss: 0.6931 - accuracy: 0.4000
Epoch 4/20
1/1 [==============================] - 0s 10ms/step - loss: 0.6914 - accuracy: 0.5000
Epoch 5/20
1/1 [==============================] - 0s 12ms/step - loss: 0.6897 - accuracy: 0.7000
Epoch 6/20
1/1 [==============================] - 0s 21ms/step - loss: 0.6880 - accuracy: 0.8000
Epoch 7/20
1/1 [==============================] - 0s 18ms/step - loss: 0.6863 - accuracy: 0.9000
Epoch 8/20
1/1 [==============================] - 0s 24ms/step - loss: 0.6846 - accuracy: 0.9000
Epoch 9/20
1/1 [==============================] - 0s 15ms/step - loss: 0.6830 - accuracy: 0.9000
Epoch 10/20
1/1 [==============================] - 0s 8ms/step - loss: 0.6813 - accuracy: 0.9000
Epoch 11/20
1/1 [============

실습| 영화 리뷰가 긍정적인지 부정적인지를 예측하기

In [10]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Embedding
from tensorflow.keras.utils import to_categorical

from numpy import array

# 텍스트 리뷰 자료를 지정한다.
docs = ["너무 재밌네요","최고예요","참 잘 만든 영화예요","추천하고 싶은 영화입니다",
        "한번 더 보고싶네요", "글쎄요", "별로예요", "생각보다 지루하네요", "연기가 어색해요",
        "재미없어요"]

# 긍정 리뷰는 1, 부정 리뷰는 0으로 클래스를 지정한다.
classes = array([1, 1, 1, 1, 1, 0, 0, 0, 0, 0])

# 토큰화
token = Tokenizer()
token.fit_on_texts(docs)
print(token.word_index)

x = token.texts_to_sequences(docs)
print('\n리뷰 텍스트, 토큰화 결과:\n', x)

# 패딩, 서로 다른 길이의 데이터를 4로 맞추어 준다.
padded_x = pad_sequences(x, 4)
print('\n패딩 결과:\n', padded_x)

# 임베딩에 입력될 단어의 수를 지정한다.
word_size = len(token.word_index) + 1

# 단어 임베딩을 포함해 딥러닝 모델을 만들고 결과를 출력한다.
model = Sequential()
model.add(Embedding(word_size, 8, input_length=4))
model.add(Flatten())
model.add(Dense(1, activation='sigmoid'))
model.summary()

model.compile(optimizer='adam', loss='binary_crossentropy',
              metrics=['accuracy'])
model.fit(padded_x, classes, epochs=20)
print("\n Accuracy: %.4f" % (model.evaluate(padded_x, classes)[1]))

{'너무': 1, '재밌네요': 2, '최고예요': 3, '참': 4, '잘': 5, '만든': 6, '영화예요': 7, '추천하고': 8, '싶은': 9, '영화입니다': 10, '한번': 11, '더': 12, '보고싶네요': 13, '글쎄요': 14, '별로예요': 15, '생각보다': 16, '지루하네요': 17, '연기가': 18, '어색해요': 19, '재미없어요': 20}

리뷰 텍스트, 토큰화 결과:
 [[1, 2], [3], [4, 5, 6, 7], [8, 9, 10], [11, 12, 13], [14], [15], [16, 17], [18, 19], [20]]

패딩 결과:
 [[ 0  0  1  2]
 [ 0  0  0  3]
 [ 4  5  6  7]
 [ 0  8  9 10]
 [ 0 11 12 13]
 [ 0  0  0 14]
 [ 0  0  0 15]
 [ 0  0 16 17]
 [ 0  0 18 19]
 [ 0  0  0 20]]
Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_2 (Embedding)     (None, 4, 8)              168       
                                                                 
 flatten_1 (Flatten)         (None, 32)                0         
                                                                 
 dense_1 (Dense)             (None, 1)                 33        
                             